In [0]:
# Setup parameters
dbutils.widgets.text("ClientContainer", "claimsprocessing", "Client Catalog Name")
client_container = dbutils.widgets.get("ClientContainer").strip()
print(f"Executing Client Pipeline for Catalog: {client_container}")

In [0]:
# Step 1: Ensure Target Database Exists
spark.sql("CREATE DATABASE IF NOT EXISTS claimsprocessing.gold")

In [0]:
# Step 2: Execute Gold Merge directly from Config & SQL scripts
import os, json

current_dir = os.getcwd()
if os.path.basename(current_dir).lower() == "dimclient":
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    config_path = os.path.join(current_dir, "Gold", "Config", "gdimclient.json")
else:
    project_root = current_dir
    config_path = os.path.join(current_dir, "dimClient", "Gold", "Config", "gdimclient.json")

print(f"Project Root: {project_root}")
print(f"Config Path: {config_path}")

with open(config_path, "r") as f:
    config_data = json.load(f)

for entity_row in config_data.get("SubLayerProcessing", []):
    entity_name = entity_row.get("SubGroupEntity")
    destination_table = entity_row.get("DestinationTable")
    
    print(f"=== Processing {entity_name} into {destination_table} ===")
    
    # Locate Source CSV File
    raw_csv_path = os.path.join(project_root, "source", "Client", "client_metadata.csv")
    print(f"Loading raw client metadata from: {raw_csv_path}")
    
    df_raw = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(raw_csv_path)
    df_raw.createOrReplaceTempView("client_raw")
    print(f"Loaded {df_raw.count()} raw client rows.")
    
    # Execute Transformation SQL Script
    sql_script_path = os.path.join(os.path.dirname(config_path), entity_row["SQLScriptPath"])
    with open(sql_script_path, "r") as sf:
        sql_query = sf.read()
    
    df_updates = spark.sql(sql_query)
    df_updates.createOrReplaceTempView("temp_updates")
    print(f"Transformation generated {df_updates.count()} records.")
    
    # Ensure Target Table Exists
    spark.sql("""
    CREATE TABLE IF NOT EXISTS claimsprocessing.gold.gold_dimclient (
     clientKey      bigint
    ,clientCode     string
    ,clientName     string
    ,subClientCode  string
    ,subClientName  string
    ) USING delta;
    """)
    
    # Execute Delta Merge SQL Script
    merge_script_path = os.path.join(os.path.dirname(config_path), entity_row["MergeScriptPath"])
    with open(merge_script_path, "r") as mf:
        merge_query = mf.read()
    
    spark.sql(merge_query)
    print(f"=== {entity_name} Gold Merge completed successfully! ===")